https://www.kaggle.com/datasets/omrathod2003/140-most-popular-crops-image-dataset?resource=download-directory&select=RGB_224x224

Dataset "Plants and Crops Image Dataset For Agriculture" (thường được biết đến với tên gọi phổ biến là tập dữ liệu 140 loại cây trồng) trên Kaggle là một tập dữ liệu hình ảnh tiêu chuẩn, rất lý tưởng cho các bài toán Thị giác máy tính (Computer Vision) và Trí tuệ nhân tạo trong nông nghiệp (Agri-AI).   

1. Tổng quan cấu trúc & Thông số kỹ thuậtSố lượng lớp (Classes): Khoảng gần 140 loại cây trồng khác nhau (ví dụ: lúa, ngô, mía, chuối, chanh, các loại đậu, hạt, rau màu, v.v.).   Số lượng ảnh mỗi lớp: Trung bình khoảng 250 ảnh/lớp, giúp cân bằng tương đối dữ liệu giữa các nhãn.Kích thước ảnh (RGB_224x224): Thư mục bạn chọn chứa các hình ảnh đã được chuẩn hóa kích thước về 224x224 pixels. Đây là kích thước tiêu chuẩn "ăn tiền" để đưa trực tiếp vào các mô hình Deep Learning phổ biến (như ResNet, EfficientNet, MobileNet, Vision Transformer...).   Định dạng màu: Ảnh chuẩn RGB, rất thuận tiện để huấn luyện ngay mà không mất công xử lý hay chuyển đổi kênh màu phức tạp.   

2. Dataset này dùng để làm gì? (Ứng dụng thực tế)Dataset này sinh ra để giải quyết các bài toán cốt lõi trong Nông nghiệp thông minh (Smart/Precision Agriculture):Phân loại cây trồng (Crop Classification): Xây dựng mô hình AI nhận diện loại cây/nông sản dựa trên hình ảnh chụp lá, thân, quả hoặc toàn bộ cây.   Hệ thống giám sát nông nghiệp tự động: Hỗ trợ máy bay không người lái (drone) hoặc camera thông minh quét và thống kê cơ cấu cây trồng trên cánh đồng.Ứng dụng di động cho nông dân: Tạo app chụp ảnh cây trồng để tra cứu thông tin sinh trưởng, mùa vụ hoặc sâu bệnh liên quan.

3. Gợi ý hướng tiếp cận khi code với Python / PyTorch / TensorFlowVì bạn có nền tảng tốt về lập trình backend và xử lý dữ liệu, việc train model với dataset này khá nhanh gọn nếu dùng cấu trúc chuẩn:Load dữ liệu: Sử dụng image_dataset_from_directory của TensorFlow/Keras hoặc ImageFolder kết hợp DataLoader của PyTorch vì cấu trúc thư mục đã được chia sẵn theo tên từng loại cây.Data Augmentation: Do mỗi class có khoảng 250 ảnh (con số vừa phải), bạn nên dùng thêm các kỹ thuật tăng cường dữ liệu (xoay ảnh, lật ngang, chỉnh độ sáng, zoom) để mô hình học tổng quát tốt hơn, tránh Overfitting.   Transfer Learning: Tận dụng các pre-trained models (như MobileNetV3 hoặc ResNet50) đóng băng các layer đầu và fine-tune lại các layer cuối để đạt độ chính xác cao nhanh chóng.

check torch o terminal

pip3 show torch
pip3 show torchvision

In [5]:
import os
import matplotlib.pyplot as plt
import random
from PIL import Image

In [9]:
dataset_path = "RGB_224x224"

In [10]:
train_dir = os.path.join(dataset_path, "train")

In [11]:
classes = os.listdir(train_dir)

FileNotFoundError: [WinError 3] The system cannot find the path specified: 'RGB_224x224\\train'

In [ ]:
len(classes)

In [ ]:
classes

In [ ]:
random.sample(classes, 5)

In [ ]:
def show_sample_images(num_classes = 4):
    fig, axes = plt.subplots(num_classes, 5, fig_size = (15,3 * num_classes))
    for i, class_name in enumerate(random.sample(classes, num_classes)):
        class_path = os.path.join(train_dir, class_name)
        images = os.listdir(class_path)

        for j in range(5):
            img_path = os.path.join(class_path, images[j])
            img = Image.open(img_path)
            axes[i, j].imshow(img)
            axes[i, j].set_title(class_name)
            axes[i, j].axis("off")
    plt.tight_layout()

In [ ]:
show_sample_images()

In [ ]:
import torch
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

In [ ]:
transform = transforms.Compose)[
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([[0.485,0.456,0.406],
    [0.229,0.224,0.225]])
]

In [ ]:
train_dataset = ImageFolder(os.path.join(dataset_path, "train"), transforms=transforms)
val_dataset = ImageFolder(os.path.join(dataset_path, "val"), transforms=transforms)
test_dataset = ImageFolder(os.path.join(dataset_path, "test"), transforms=transforms)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=8)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [ ]:
import torchvision.models as models
import torch.nn as nn

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
torch.backends.mps.is_available()

In [ ]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

In [ ]:
device

In [ ]:
model = models.resnet18(pretrained = True)

In [ ]:
for param in model.parameters():
    param.requires_grad = False

In [ ]:
num_classes = len(train_dataset.classes)
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)

In [ ]:
import torch.optim as optim
import torch.nn.functional as F

In [ ]:
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

In [ ]:
def train_one_epoch(model, loader):
    model.train()
    running_loss = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_glad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss == loss.item()

    return running_loss / len(loader)

In [ ]:
def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().items()

    return correct/total

In [ ]:
epochs = 5
for epoch in range(epochs):
    train_loss = train_one_epoch(model, train_loader)
    val_acc =evaluate(model, val_loader)
    print(f"Epoch {epoch +1}, Train Loss: {train_loss:.4f}, Val Accuracy: {val_acc:.4f}")

In [ ]:
test_acc =evaluate(model, test_loader)

In [ ]:
print(f"Test Accuracy: {test_acc: .4f}")

In [ ]:
def visualize_predictions(model, loader, n=5):
    model.eval()
    images_shown = 0
    class_names = train_dataset.class_names

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)

            for i in range(n):
                img = images[i].cpu().permute(1,2,0).numpy()
                plt.imshow((img* 0.229 + 0.485))
                plt.title(f"True: {class_names[labels[i]]}, Pred: {class_names[preds[i]]}")
                plt.axis("off")
                plt.show()

                images_shown += 1

                if images_shown >=n:
                    return

In [ ]:
visualize_predictions(model, test_loader, 3)

In [ ]:
import joblib

In [ ]:
model_data = {
    "model_state_dict": model.state_dict(),
    "class_to_id": train_dataset.class_to_id
}

In [ ]:
model_data

In [ ]:
joblib.dump(model_data, "Crop_classifier_model.pkl")

In [ ]:
https://colab.research.google.com/drive/11v6zJP7B0W4-5v8auocZkJMV7I_KowiJ#scrollTo=947e8045